# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset using the `mlcroissant` library. All dataset entities are referenced via their Croissant `@id` fields for reliability and reproducibility.

### Dataset Source
The dataset is specified via a publicly accessible Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records from the Croissant schema using `mlcroissant`. This section reads metadata and gives a dataset overview.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the Croissant URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Print metadata summary
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")
print(f"Published: {getattr(meta, 'datePublished', 'N/A')}")

## 2. Data Overview
Review available record sets and their fields/columns using their `@id`s. 
Let's list all record sets, inspect their field `@id`s, and show example records for each.

In [ ]:
# Gather all record set IDs from the dataset
record_sets = [rs['@id'] for rs in getattr(dataset.metadata, 'recordSet', [])]
if len(record_sets) == 0:
    print('No explicit record sets found in metadata. Attempting to infer via dataset API...')
    # List possible record sets
    record_sets = dataset.list_record_sets()
print('Available Record Sets:')
for idx, rs in enumerate(record_sets):
    print(f'  {idx+1}. @id={rs}')

# For each record set, print the available fields (by @id)
for rs in record_sets:
    print(f'\nRecord Set @id: {rs}')
    fields = dataset.list_fields(record_set=rs)
    print('  Fields:')
    for f in fields:
        print(f'    - {f}')
    # Show up to 2 records as example
    print('  Example records:')
    ex = list(dataset.records(record_set=rs))
    for i, rec in enumerate(ex[:2]):
        print(f'    [{i+1}]:')
        pprint(rec)


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s. 
Typically, the main record set contains protein-level data. For this dataset, we use the record set identified from the previous overview.

In [ ]:
# Specify the main record set by @id (e.g., 'https://api.app.sen.science/frontiers/7154140/03bd4e3c-a5fc-4362-8a42-766cbf6be82e')
main_record_set_id = record_sets[0] if record_sets else None
print(f"Using record set: {main_record_set_id}")

# Load all records from this record set into a DataFrame
records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)

# Show the fields/columns present
print('Available columns (by @id):')
print(df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
Let's perform typical data processing steps, such as filtering, normalization, and group analysis on the protein record set. 

For demonstration, we'll use the field `cr:PeptideCount` (replace with your dataset's peptide count field `@id` if different) as the numeric target, and group by protein accession if available.

In [ ]:
# You may need to adjust these @id values based on your dataset inspection above:
numeric_field = None
group_field = None

# Try to select a likely peptide count or abundance field
for col in df.columns:
    if 'peptide' in col.lower() or 'abund' in col.lower():
        numeric_field = col
    if 'accession' in col.lower() or 'name' in col.lower():
        group_field = col
print(f"Chosen numeric field: {numeric_field}\nChosen group field: {group_field}")

# For safety, coerce to numeric and drop missing
df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')

threshold = df[numeric_field].quantile(0.75)  # example: filter to upper quartile
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.2f} (upper quartile):")
print(filtered_df[[numeric_field, group_field]].head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized '{numeric_field}' for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

if group_field and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped mean {numeric_field} by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions such as the histogram of peptide counts and optionally the normalized values. 
We'll use matplotlib for plotting.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(7, 4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.show()

# If normalization performed
if f"{numeric_field}_normalized" in filtered_df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(filtered_df[f"{numeric_field}_normalized"], bins=30, kde=True)
    plt.title(f"Normalized Distribution of {numeric_field} (filtered)")
    plt.xlabel(f"{numeric_field}_normalized")
    plt.ylabel('Frequency')
    plt.show()


## 6. Conclusion
In this notebook, we used the `mlcroissant` library to programmatically access a mass spectrometry dataset using its Croissant schema, referenced all elements by their `@id`, and performed basic EDA. 

- The dataset features protein records with multiple quantitative and categorical fields.
- Record sets and fields can be reliably referenced by `@id` for robust dataset handling.
- Filtering and normalization steps reveal distributional properties of key numeric fields (such as peptide count or abundances).

Explore further by joining to record sets, analyzing other fields, or integrating protein annotations from external resources!

_Notebook generated using `mlcroissant` and dataset at [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)_